# Feature Engineering
Build `team_features.csv` — the final modeling dataset used by the regression and trajectory models.

In [ ]:
import pandas as pd
import numpy as np

teams    = pd.read_csv('../data_clean/teams_clean.csv')
games    = pd.read_csv('../data_clean/games_clean.csv', parse_dates=['Date'])
schedule = pd.read_csv('../data_raw/remaining_schedule.csv')

print(teams.columns.tolist())

## 1. Home / Away Record

In [ ]:
home = games.groupby('home_team').agg(
    home_wins   = ('home_win', 'sum'),
    home_games  = ('home_win', 'count'),
).rename_axis('Team')

away = games.groupby('away_team').agg(
    away_wins  = ('home_win', lambda x: (1 - x).sum()),
    away_games = ('home_win', 'count'),
).rename_axis('Team')

splits = home.join(away, how='outer').fillna(0)
splits['home_win_pct'] = splits['home_wins'] / splits['home_games']
splits['away_win_pct'] = splits['away_wins'] / splits['away_games']
splits.head()

## 2. Recent Form (last 10 games)

In [ ]:
records = []
for _, row in games.iterrows():
    records.append({'team': row['home_team'], 'date': row['Date'], 'win': int(row['home_win']),   'margin': row['margin']})
    records.append({'team': row['away_team'], 'date': row['Date'], 'win': int(1 - row['home_win']), 'margin': -row['margin']})

log = pd.DataFrame(records).sort_values(['team', 'date'])

recent = log.groupby('team').tail(10).groupby('team').agg(
    last10_wins   = ('win', 'sum'),
    last10_margin = ('margin', 'mean'),
).rename_axis('Team')

recent['last10_win_pct'] = recent['last10_wins'] / 10
recent.head()

## 3. Remaining Strength of Schedule

In [ ]:
win_pct_map = teams.set_index('Team')['win_pct']

def sos_remaining(team):
    opps = pd.concat([
        schedule.loc[schedule['away_team'] == team, 'home_team'],
        schedule.loc[schedule['home_team'] == team, 'away_team'],
    ])
    return opps.map(win_pct_map).mean()

games_left = (
    pd.concat([schedule['away_team'], schedule['home_team']])
    .value_counts()
    .rename('games_remaining')
    .rename_axis('Team')
)

sos_df = pd.DataFrame({'sos_remaining': {t: sos_remaining(t) for t in teams['Team']}})
sos_df.index.name = 'Team'
sos_df.head()

## 4. Projected Wins (blended pace model)

In [ ]:
teams_idx = teams.set_index('Team')

features = teams_idx.join(splits, how='left') \
                     .join(recent, how='left') \
                     .join(sos_df, how='left') \
                     .join(games_left, how='left')

features['games_remaining'] = features['games_remaining'].fillna(0)

# Blended rate: 70% season win% + 30% last-10 win%
features['blended_rate'] = (
    0.7 * features['win_pct'] +
    0.3 * features['last10_win_pct'].fillna(features['win_pct'])
)
features['projected_wins'] = (
    features['wins'] + features['blended_rate'] * features['games_remaining']
).round(1)

features = features.sort_values('projected_wins', ascending=False)
features[['wins', 'projected_wins', 'last10_wins', 'sos_remaining', 'NRtg']].head(10)

## 5. Save

In [ ]:
features.reset_index().to_csv('../data_clean/team_features.csv', index=False)
print('Saved team_features.csv with', features.shape[1], 'features')
print(features.columns.tolist())